# EDA de Áudio — Reconhecimento de Voz com MFCC

Análise exploratória dos áudios em `audio_exemplo/` preparando o terreno para um pipeline de reconhecimento de voz.

**Pipeline previsto:**
1. Carregamento → 2. Subamostragem (16 kHz) → 3. Filtro passa-banda de voz → 4. Extração de MFCC → 5. Classificação

## 1. Setup e Imports

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import scipy.signal as signal
import soundfile as sf
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 4),
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.size': 10,
})

AUDIO_DIR = Path('audio_exemplo')
TARGET_SR = 16000  # SR padrão para processamento de voz

print(f'Diretório de áudio: {AUDIO_DIR.resolve()}')

## 2. Carregamento dos Áudios

In [ ]:

audio_files = sorted(AUDIO_DIR.glob('*'))
audios = {}

# Profundidade de bits por subtype (o que o firmware gravou; o INMP441 entrega 24 bits via I2S)
BITS_BY_SUBTYPE = {'PCM_16': 16, 'PCM_24': 24, 'PCM_32': 32,
                   'FLOAT': 32, 'DOUBLE': 64}

for fpath in audio_files:
    if fpath.suffix.lower() not in ('.wav', '.ogg', '.mp3', '.flac'):
        continue
    m = sf.info(str(fpath))
    y, sr = librosa.load(fpath, sr=None, mono=True)
    name = fpath.stem

    # Diagnóstico rápido adaptado ao mic: DC offset (I2S), clipagem, profundidade
    dc_offset    = float(y.mean())
    peak         = float(np.max(np.abs(y))) if len(y) else 0.0
    clipping_pct = float((np.abs(y) > 0.99).mean() * 100) if len(y) else 0.0
    bits         = BITS_BY_SUBTYPE.get(m.subtype, np.nan)

    # Convenção de nome do plano (§2): locutor_<nome>_<sessao>.wav
    parts = name.split('_')
    if parts[0] == 'locutor' and len(parts) >= 3:
        speaker, session = parts[1], parts[2]
    else:
        speaker, session = None, None

    audios[name] = {
        'y_original': y,
        'sr_original': sr,
        'duration': len(y) / sr,
        'samples': len(y),
        'file': fpath.name,
        'subtype': m.subtype,
        'bit_depth': bits,
        'dc_offset': dc_offset,
        'peak': peak,
        'clipping_pct': clipping_pct,
        'speaker': speaker,
        'session': session,
    }
    print(f'{fpath.name:30s} SR={sr:>6} Hz  {m.subtype:<8s} dur={len(y)/sr:5.2f}s  '
          f'peak={peak:+.2f}  DC={dc_offset:+.4f}  clip={clipping_pct:.2f}%')


## 2.1 Diagnóstico do sinal

O pipeline resampleia tudo para 16 kHz (`TARGET_SR`), então qualquer taxa que o firmware
entregue funciona. Este bloco só avisa sobre **DC offset** ou **clipagem** — sintomas comuns
de gravação I2S/ganho mal configurados.


In [ ]:
print(f'TARGET_SR do pipeline = {TARGET_SR} Hz (resample automático, se preciso)\n')

for name, info in audios.items():
    sr = info['sr_original']
    status_sr = 'nativo' if sr == TARGET_SR else f'resample {sr} -> {TARGET_SR} Hz'
    alertas = []
    if np.abs(info['dc_offset']) > 5e-3:
        alertas.append('DC offset alto')
    if info['clipping_pct'] > 0.1:
        alertas.append(f"clipagem {info['clipping_pct']:.1f}%")
    print(f'  {name:24s} SR={sr:>7} Hz ({status_sr})  bits={info["bit_depth"]}  '
          f'{" | ".join(alertas) if alertas else "OK"}')


## 3. Onda Sonora — Sinal Original

In [ ]:
fig, axes = plt.subplots(len(audios), 1, figsize=(14, 3.5 * len(audios)), sharex=False)
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    t = np.arange(len(info['y_original'])) / info['sr_original']
    ax.plot(t, info['y_original'], linewidth=0.3, color='steelblue')
    ax.set_title(f"{info['file']}  |  SR={info['sr_original']} Hz  |  {info['duration']:.2f}s")
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, info['duration'])

axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('Ondas Sonoras — Sinal Original', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Subamostragem para 16 kHz

16 kHz é o sampling rate padrão para processamento de fala (cobrindo até 8 kHz — bem acima da faixa de voz humana fundamental).

In [ ]:
for name, info in audios.items():
    if info['sr_original'] != TARGET_SR:
        info['y_resampled'] = librosa.resample(
            info['y_original'], orig_sr=info['sr_original'], target_sr=TARGET_SR
        )
        print(f"{name}: {info['sr_original']} -> {TARGET_SR} Hz  |  {info['samples']} -> {len(info['y_resampled'])} amostras")
    else:
        info['y_resampled'] = info['y_original'].copy()
        print(f'{name}: já está em {TARGET_SR} Hz')

    info['sr'] = TARGET_SR
    info['y'] = info['y_resampled']

In [ ]:
fig, axes = plt.subplots(len(audios), 1, figsize=(14, 3.5 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    t = np.arange(len(info['y'])) / TARGET_SR
    ax.plot(t, info['y'], linewidth=0.3, color='darkorange')
    ax.set_title(f"{name}  |  Subamostrado p/ {TARGET_SR} Hz  |  {len(info['y'])/TARGET_SR:.2f}s")
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, len(info['y']) / TARGET_SR)

axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('Ondas Sonoras — Subamostrado (16 kHz)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Filtro Passa-Banda de Voz

A voz humana fundamental fica entre ~85 Hz e ~300 Hz, com harmônicos relevantes até ~8 kHz. Um filtro passa-banda **80–8000 Hz** retém toda a informação de voz eliminando ruído de baixa e alta frequência.

In [ ]:
VOICE_LOW = 80
VOICE_HIGH = 8000

for name, info in audios.items():
    sr = info['sr']
    nyq = sr / 2.0
    low = VOICE_LOW / nyq
    high = min(VOICE_HIGH / nyq, 0.99)

    b, a = signal.butter(N=5, btype='bandpass', Wn=[low, high])
    info['y_filtered'] = signal.filtfilt(b, a, info['y'])

    print(f"{name}: filtro passa-banda [{VOICE_LOW}-{VOICE_HIGH}] Hz aplicado (butterworth ordem 5)")

In [ ]:
fig, axes = plt.subplots(len(audios), 1, figsize=(14, 3.5 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    t = np.arange(len(info['y'])) / TARGET_SR
    ax.plot(t, info['y'], linewidth=0.3, color='steelblue', alpha=0.5, label='Original (16kHz)')
    ax.plot(t, info['y_filtered'], linewidth=0.3, color='crimson', alpha=0.8, label=f'Filtrado [{VOICE_LOW}-{VOICE_HIGH}] Hz')
    ax.set_title(f"{name}")
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, len(info['y']) / TARGET_SR)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('Comparação: Antes vs Depois do Filtro Passa-Banda de Voz', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Resposta em Frequência do Filtro

In [ ]:
nyq = TARGET_SR / 2.0
b, a = signal.butter(N=5, btype='bandpass', Wn=[VOICE_LOW / nyq, min(VOICE_HIGH / nyq, 0.99)])
w, h = signal.freqz(b, a, worN=2048, fs=TARGET_SR)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

ax1.plot(w, 20 * np.log10(np.abs(h)), color='steelblue', linewidth=1.5)
ax1.axvline(VOICE_LOW, color='crimson', linestyle='--', alpha=0.7, label=f'{VOICE_LOW} Hz')
ax1.axvline(VOICE_HIGH, color='crimson', linestyle='--', alpha=0.7, label=f'{VOICE_HIGH} Hz')
ax1.set_title('Resposta em Amplitude do Filtro Butterworth (ordem 5)')
ax1.set_xlabel('Frequência (Hz)')
ax1.set_ylabel('Ganho (dB)')
ax1.set_xlim(0, 10000)
ax1.set_ylim(-80, 5)
ax1.legend()

ax2.plot(w, np.unwrap(np.angle(h)), color='darkorange', linewidth=1.5)
ax2.set_title('Resposta em Fase')
ax2.set_xlabel('Frequência (Hz)')
ax2.set_ylabel('Fase (rad)')
ax2.set_xlim(0, 10000)

plt.tight_layout()
plt.show()

## 7. Espectro de Frequência (FFT)

In [ ]:
fig, axes = plt.subplots(len(audios), 1, figsize=(14, 3.5 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    sr = info['sr']
    y_filt = info['y_filtered']
    n = len(y_filt)
    Y = np.fft.rfft(y_filt)
    freqs = np.fft.rfftfreq(n, d=1/sr)
    magnitude = np.abs(Y) / n

    ax.plot(freqs, 20 * np.log10(magnitude + 1e-10), linewidth=0.4, color='forestgreen')
    ax.set_title(f"{name} — Espectro de Frequência (após filtro)")
    ax.set_ylabel('Magnitude (dB)')
    ax.set_xlim(0, sr / 2)
    ax.set_ylim(-80, 0)

axes[-1].set_xlabel('Frequência (Hz)')
fig.suptitle('Espectro de Frequência — Sinal Filtrado', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Espectrograma de Potência (STFT)

In [ ]:
N_FFT = 1024
HOP_LENGTH = 256  # 16ms com SR=16kHz — bom para voz

fig, axes = plt.subplots(len(audios), 1, figsize=(14, 4 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    S = np.abs(librosa.stft(info['y_filtered'], n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
    S_db = librosa.power_to_db(S, ref=np.max)

    img = librosa.display.specshow(
        S_db, sr=info['sr'], hop_length=HOP_LENGTH,
        x_axis='time', y_axis='hz', ax=ax, cmap='magma'
    )
    ax.set_title(f"{name} — Espectrograma de Potência (n_fft={N_FFT}, hop={HOP_LENGTH})")
    ax.set_ylim(0, 8000)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

fig.suptitle('Espectrograma — Sinal Filtrado na Faixa de Voz', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Espectrograma Mel

A escala Mel aproxima a percepção auditiva humana — essencial para extração de MFCC.

In [ ]:
N_MELS = 128

fig, axes = plt.subplots(len(audios), 1, figsize=(14, 4 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    S_mel = librosa.feature.melspectrogram(
        y=info['y_filtered'], sr=info['sr'],
        n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    S_mel_db = librosa.power_to_db(S_mel, ref=np.max)

    img = librosa.display.specshow(
        S_mel_db, sr=info['sr'], hop_length=HOP_LENGTH,
        x_axis='time', y_axis='mel', ax=ax, cmap='viridis'
    )
    ax.set_title(f"{name} — Espectrograma Mel ({N_MELS} bandas)")
    ax.set_ylim(0, 8000)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

fig.suptitle('Espectrograma Mel — Sinal Filtrado', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. MFCC — Coeficientes Cepstrais em Frequências Mel

MFCC é a feature padrão para reconhecimento de voz. Usaremos **13 coeficientes** (o mínimo recomendado para ASR), incluindo o coeficiente de energia (C0).

In [ ]:
N_MFCC = 13

fig, axes = plt.subplots(len(audios), 1, figsize=(14, 3.5 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    mfccs = librosa.feature.mfcc(
        y=info['y_filtered'], sr=info['sr'],
        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    info['mfcc'] = mfccs

    img = librosa.display.specshow(
        mfccs, sr=info['sr'], hop_length=HOP_LENGTH,
        x_axis='time', ax=ax, cmap='coolwarm'
    )
    ax.set_title(f"{name} — {N_MFCC} MFCCs")
    ax.set_ylabel('Coeficiente')
    fig.colorbar(img, ax=ax)

axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('MFCC — Sinal Filtrado na Faixa de Voz', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. Delta e Delta-Delta (Δ e ΔΔ)

As derivadas de primeira e segunda ordem capturam a dinâmica temporal dos MFCCs — cruciais para distinguir fonemas.

In [ ]:
fig, axes = plt.subplots(len(audios), 3, figsize=(18, 4 * len(audios)))
if len(audios) == 1:
    axes = [axes]

features_label = ['MFCC', 'Δ MFCC', 'ΔΔ MFCC']

for i, (name, info) in enumerate(audios.items()):
    mfcc = info['mfcc']
    delta_mfcc = librosa.feature.delta(mfcc)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    info['mfcc_delta'] = delta_mfcc
    info['mfcc_delta2'] = delta2_mfcc
    info['mfcc_full'] = np.vstack([mfcc, delta_mfcc, delta2_mfcc])

    for j, (feat, label) in enumerate(zip([mfcc, delta_mfcc, delta2_mfcc], features_label)):
        ax = axes[i][j]
        img = librosa.display.specshow(
            feat, sr=info['sr'], hop_length=HOP_LENGTH,
            x_axis='time', ax=ax, cmap='coolwarm'
        )
        ax.set_title(f"{name} — {label}")
        ax.set_ylabel('Coef.')
        fig.colorbar(img, ax=ax)

axes[-1][0].set_xlabel('Tempo (s)')
axes[-1][1].set_xlabel('Tempo (s)')
axes[-1][2].set_xlabel('Tempo (s)')
fig.suptitle('MFCC + Δ + ΔΔ — 39 features por frame (13 × 3)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 12. Estatísticas Descritivas dos MFCCs

Para cada coeficiente, calculamos média e desvio padrão ao longo do tempo — uma forma compacta de representar o "perfil" de cada locutor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['steelblue', 'crimson', 'forestgreen']

for (name, info), color in zip(audios.items(), colors):
    mfcc_mean = np.mean(info['mfcc'], axis=1)
    mfcc_std = np.std(info['mfcc'], axis=1)
    coeffs = np.arange(1, N_MFCC + 1)

    axes[0].errorbar(coeffs, mfcc_mean, yerr=mfcc_std, fmt='o-',
                     label=name, color=color, capsize=3, linewidth=1.5)
    axes[1].bar(coeffs + (0.2 * list(audios.keys()).index(name) - 0.1),
                mfcc_mean, width=0.2, label=name, color=color, alpha=0.8)

axes[0].set_title('Média ± DP dos MFCCs por Coeficiente')
axes[0].set_xlabel('Coeficiente MFCC')
axes[0].set_ylabel('Valor')
axes[0].set_xticks(range(1, N_MFCC + 1))
axes[0].legend()

axes[1].set_title('Média dos MFCCs — Comparação entre Locutores')
axes[1].set_xlabel('Coeficiente MFCC')
axes[1].set_ylabel('Valor Médio')
axes[1].set_xticks(range(1, N_MFCC + 1))
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Extração do Feature Vector Completo

Para cada arquivo, o vetor final tem shape `(39, T)` onde 39 = 13 MFCC + 13 Δ + 13 ΔΔ.

In [ ]:
print(f'{"Arquivo":25s}  {"MFCC shape":>20s}  {"Δ":>14s}  {"ΔΔ":>14s}  {"Feature Vector":>20s}')
print('-' * 100)

for name, info in audios.items():
    mfcc_shape = str(info['mfcc'].shape)
    delta_shape = str(info['mfcc_delta'].shape)
    delta2_shape = str(info['mfcc_delta2'].shape)
    full_shape = str(info['mfcc_full'].shape)
    print(f'{name:25s}  {mfcc_shape:>20s}  {delta_shape:>14s}  {delta2_shape:>14s}  {full_shape:>20s}')

print(f'\n→ Cada frame tem 39 features (13 MFCC + 13 Δ + 13 ΔΔ)')
print(f'→ hop_length={HOP_LENGTH} @ {TARGET_SR}Hz = {HOP_LENGTH/TARGET_SR*1000:.0f}ms por frame')

## 14. Distribuição dos Coeficientes MFCC (Violin Plot)

In [ ]:
fig, axes = plt.subplots(len(audios), 1, figsize=(14, 4 * len(audios)))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    mfcc_data = info['mfcc']  # (13, T)
    parts = ax.violinplot(
        [mfcc_data[i] for i in range(N_MFCC)],
        positions=range(1, N_MFCC + 1),
        showmeans=True, showmedians=True
    )
    ax.set_title(f"{name} — Distribuição dos 13 MFCCs")
    ax.set_xlabel('Coeficiente MFCC')
    ax.set_ylabel('Valor')
    ax.set_xticks(range(1, N_MFCC + 1))

fig.suptitle('Violin Plots dos MFCCs por Locutor', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 15. Mapa de Correlação entre Coeficientes MFCC

In [ ]:
fig, axes = plt.subplots(1, len(audios), figsize=(7 * len(audios), 6))
if len(audios) == 1:
    axes = [axes]

for ax, (name, info) in zip(axes, audios.items()):
    corr = np.corrcoef(info['mfcc'])
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_title(f"{name} — Correlação MFCC")
    ax.set_xlabel('Coeficiente')
    ax.set_ylabel('Coeficiente')
    ax.set_xticks(range(N_MFCC))
    ax.set_yticks(range(N_MFCC))
    fig.colorbar(im, ax=ax)

fig.suptitle('Mapa de Correlação entre Coeficientes MFCC', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 16. Resumo — Parâmetros do Pipeline

Resumo dos parâmetros usados nesta EDA, que devem ser mantidos no pipeline final.

In [ ]:
# Resumo dos parâmetros do pipeline (mantidos no modelo final)
PIPELINE = {
    'TARGET_SR': TARGET_SR,          # 16 kHz mono
    'VOICE_BAND': (VOICE_LOW, VOICE_HIGH),  # 80-8000 Hz, Butterworth ordem 5
    'N_FFT': N_FFT, 'HOP_LENGTH': HOP_LENGTH,  # 1024 / 256 = 16 ms por frame
    'N_MFCC': N_MFCC,                # 13 + Δ + ΔΔ = 39 dims/frame
    'N_MELS': N_MELS,
}
for k, v in PIPELINE.items():
    print(f'{k:12s} = {v}')
print(f"\nArquivos: {len(audios)} | convenção: locutor_<nome>_<sessao>.wav")


In [ ]:
for name, info in audios.items():
    del info['y_original']
    del info['y_resampled']
    del info['y']
    del info['y_filtered']
    del info['mfcc']
    del info['mfcc_delta']
    del info['mfcc_delta2']
    del info['mfcc_full']

print('Variáveis intermediárias limpas.')
print('Dados disponíveis em audios[name] com chaves: sr, sr_original, duration, samples, file, '
      'subtype, bit_depth, dc_offset, peak, clipping_pct, speaker, session')